# CS-421 Machine learning for behavioral data
## Project - GoGymi dataset
### Fatumah binta Doukouré 340969 - Louis Tschanz 315774 - Majandra Garcia 347470

---

**Main research question:** Which observable student behaviors, including written response quality, chatbot interaction patterns, and early platform engagement, are most predictive of academic success?

- **Sub-question 1:** How do the linguistic and semantic characteristics of students' written responses relate to their performance across essay and text comprehension assessments?

- **Sub-question 2:** To what extent does chatbot (GymiTrainer) engagement, in terms of : frequency, interaction intensity, and feedback patterns, influence student performance in quizzes and essays?

- **Sub-question 3:** To what extent can early learning behaviors, such as exercise diversity, session consistency, and content focus, predict which students will struggle or succeed later in their learning progression?

---


## Dataset Description

### Main parts of the dataset

- **User tables**
  - `students.csv`, `teachers.csv`
  - user identifiers, group membership, account creation time

- **Platform activity**
  - `pageviews.csv`
  - `events/*.csv`
  - navigation and fine-grained interaction logs (clicks, scroll, question views, media events, chatbot open/close)

- **Chatbot usage**
  - `gymitrainer.csv`
  - `gymitrainer_feedback.csv`
  - conversation threads with the AI chatbot and user feedback on those interactions

- **Performance / outcome tables**
  - `quiz_results.csv`
  - `math_results.csv`
  - `text_results.csv`
  - `essay_results.csv`
  - `essay_feedback.csv`

- **Supporting / mapping tables**
  - `course_ids.csv`
  - `math_questions.csv`
  - `quiz_questions.csv`
  - `text_questions.csv`
  - `comments.csv`

### Most important identifiers

The main linking variable across the dataset is usually:

- `user_id`

Other useful keys include:

- `result_id` for essay submissions
- `thread_id` for chatbot feedback
- `question_id` for question-level results
- `url` / `post_id` for linking pages to course content

### Practical note

The dataset mixes several time formats:
- Unix timestamps in **seconds**
- client-side event timestamps in **milliseconds**
- datetime strings in `pageviews.csv`

---

#### Useful datasets per sub-question:

**Sub-question 1**
- essay_results : the essay text + all AI dimension scores are your core data
- essay_feedback : teacher grades give you a ground truth label
- text_results : student answer text + points
- text_questions : question metadata to contextualize answers
- course_ids : to distinguish Langzeit vs Kurzzeitgymnasium

**Sub-question 2**
- gymitrainer : conversation content, frequency, thread length
- gymitrainer_feedback : user satisfaction scores
- events_gymitrainer : open/close timestamps give you session duration
- quiz_results : outcome to predict
- essay_results : second outcome
- pageviews : useful to establish when a student visited a page relative to chatbot use

**Sub-question 3**
- events_heartbeat : session consistency, time on platform
- events_questions : exercise diversity
- quiz_results : captures progression over time
- math_results : another longitudinal performance signal
- pageviews : content focus (which topics visited)
- students : group membership to control for classroom effects

---

## Sub-question 1

#### How do the linguistic and semantic characteristics of students' written responses relate to their performance across essay and text comprehension assessments?

In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [30]:
# Directory
print("Python working directory:", os.getcwd())

data_dir = './data'
assert os.path.isdir(data_dir), f"File not found: {data_dir}" # check files exist

Python working directory: c:\Users\msgar\OneDrive\Documents\EPFL\Cours\MA4\ML4BD\MLBD_2026


## Loading the Data by type

In [31]:
# 1. User Tables
students = pd.read_csv('{}/students.csv'.format(data_dir))
teachers = pd.read_csv('{}/teachers.csv'.format(data_dir))

In [32]:
# 2. Platform activity tables
pageviews = pd.read_csv('{}/pageviews.csv'.format(data_dir))
## events will do later

In [33]:
# 3. Chatbots specific tables
gymitrainer_feedback = pd.read_csv('{}/gymitrainer_feedback.csv'.format(data_dir))
gymitrainer = pd.read_csv('{}/gymitrainer.csv'.format(data_dir))

In [34]:
# 4. Learning outcomes tables / Perforamance tables
math_results = pd.read_csv('{}/math_results.csv'.format(data_dir))
quiz_results = pd.read_csv('{}/quiz_results.csv'.format(data_dir))
text_results = pd.read_csv('{}/text_results.csv'.format(data_dir))
essay_results = pd.read_csv('{}/essay_results.csv'.format(data_dir))

In [35]:
# 5. Supporting / Mapping / metadata tables // help connect the other tables
course_ids = pd.read_csv('{}/course_ids.csv'.format(data_dir))
math_questions = pd.read_csv('{}/math_questions.csv'.format(data_dir))
quiz_questions = pd.read_csv('{}/quiz_questions.csv'.format(data_dir))
text_questions = pd.read_csv('{}/text_questions.csv'.format(data_dir))
essay_feedback = pd.read_csv('{}/essay_feedback.csv'.format(data_dir))
comments = pd.read_csv('{}/comments.csv'.format(data_dir))

## Creating the Tables

In [36]:
print("Number of distincts users in math submissions :", math_results['user_id'].nunique(),"users")
print("Number of distincts users in quiz submissions :", quiz_results['user_id'].nunique(),"users")
print("Number of distincts users in text submissions :", text_results['user_id'].nunique(),"users")
print("Number of distincts users in essay submissions :", essay_results['user_id'].nunique(),"users")

Number of distincts users in math submissions : 227 users
Number of distincts users in quiz submissions : 1606 users
Number of distincts users in text submissions : 647 users
Number of distincts users in essay submissions : 1402 users


In [37]:
math_results.head()

,Unnamed: 0,session_id,question_part,user_id,question_id,points,max_points,answer,timestamp
0,0,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63e3,0.0,4.0,NaN,1738158044
1,1,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63e1,0.0,2.0,NaN,1738158044
2,2,679a2fdce127578405afeef5,1,490,66a5eb558948dda8b2fd63e1,0.0,2.0,NaN,1738158044
3,3,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63df,4.0,4.0,21,1738158044
4,4,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63e0,0.0,4.0,NaN,1738158044


In [38]:
perf_by_users = pd.DataFrame({
    'user_id': pd.unique(
        pd.concat([math_results['user_id'], text_results['user_id'],quiz_results['user_id'],essay_results['user_id']], ignore_index=True)
    )
})

perf_by_users = perf_by_users.sort_values('user_id').reset_index(drop=True)
perf_by_users.head()

,user_id
0,1
1,4
2,5
3,6
4,7


In [39]:
# Compute absolute points column
avg_point_quiz_absolute = quiz_results.groupby('user_id')['points'].mean().reset_index(name='average_point_quiz_absolute')

# Compute relative points column
# Fix inconsistent rows (Hint 2)
quiz_results.loc[quiz_results['points'] > quiz_results['max_points'], 'max_points'] = \
    quiz_results.loc[quiz_results['points'] > quiz_results['max_points'], 'points']

# Fix when max points = 0
quiz_valid = quiz_results[quiz_results['max_points'] > 0].copy() # remove questions if max_point=0, it means they don't count
quiz_valid['points_relative'] = quiz_valid['points'] / quiz_valid['max_points'] # compute relative score
avg_point_quiz_relative = (quiz_valid.groupby('user_id')['points_relative'].mean().reset_index(name='average_point_quiz_relative'))

# Merge absolute quiz points
perf_by_users = perf_by_users.merge(avg_point_quiz_absolute, on='user_id', how='left')

# Merge relative quiz points
perf_by_users = perf_by_users.merge(avg_point_quiz_relative, on='user_id', how='left')

In [40]:
perf_by_users.sample(10)

,user_id,average_point_quiz_absolute,average_point_quiz_relative
1758,6462,0.939394,0.939394
567,1114,0.537879,0.544061
95,266,NaN,NaN
1649,6320,0.352941,0.352941
1255,1978,0.854369,0.854369
367,829,0.783582,0.783582
902,1523,0.661721,0.671687
319,735,0.825073,0.837278
1357,3191,NaN,NaN
2082,7072,0.672566,0.672566


pk tu check sur math_results juste après, alors que tu utilises quiz_results avant?

In [41]:
# check if user_id is in math_results
user_id_to_check = 491  # replace with the user id you want to inspect

if user_id_to_check in math_results['user_id'].values:
    print(f"{user_id_to_check} is in math_results")
    display(math_results[math_results['user_id'] == user_id_to_check].head())
else:
    print(f"{user_id_to_check} is not in math_results")

491 is not in math_results


Maths scores with math_results

In [42]:
# Fix inconsistent rows
math_results.loc[math_results['points'] > math_results['max_points'], 'max_points'] = \
    math_results.loc[math_results['points'] > math_results['max_points'], 'points']

# Fix when max points = 0
math_valid = math_results[math_results['max_points'] > 0].copy()
math_valid['points_relative'] = math_valid['points'] / math_valid['max_points']

# Absolute and relative scores per user
avg_point_math_absolute = math_valid.groupby('user_id')['points'].mean().reset_index(name='average_point_math_absolute')
avg_point_math_relative = math_valid.groupby('user_id')['points_relative'].mean().reset_index(name='average_point_math_relative')

# Merge into perf_by_users
perf_by_users = perf_by_users.merge(avg_point_math_absolute, on='user_id', how='left')
perf_by_users = perf_by_users.merge(avg_point_math_relative, on='user_id', how='left')

perf_by_users.sample(10)

,user_id,average_point_quiz_absolute,average_point_quiz_relative,average_point_math_absolute,average_point_math_relative
1112,1803,0.217391,0.218447,NaN,NaN
495,1015,NaN,NaN,NaN,NaN
19,58,0.828947,0.828947,NaN,NaN
677,1256,0.522727,0.526718,NaN,NaN
898,1519,0.916667,0.916667,NaN,NaN
2060,7046,0.686047,0.686047,NaN,NaN
2071,7059,0.797297,0.797297,NaN,NaN
380,843,0.480159,0.480000,NaN,NaN
130,317,NaN,NaN,NaN,NaN
1825,6544,0.866667,0.866667,NaN,NaN


Essay scores (aggregating the multiple scoring dimensions from essay_results)

In [43]:
# Compute essay scores per user
# Fix inconsistent rows
essay_results_clean = essay_results.copy()

# Define all scoring dimensions
essay_score_cols = [
    'content__on_topic', 'content__plausible', 'content__convincing_ideas', 'content__scope',
    'structure__coherence', 'structure__outline', 'structure__repetition', 'structure__text_pattern',
    'language__clarity', 'language__spelling', 'language__puncutation', 'language__word_choice',
    'language__sentence_structure', 'language__style'
]

# Compute average across all dimensions per submission
essay_results_clean['essay_score_mean'] = essay_results_clean[essay_score_cols].mean(axis=1)

# Aggregate per user: average overall score
avg_essay_score = essay_results_clean.groupby('user_id')['essay_score_mean'].mean().reset_index(name='average_essay_score')

# Number of essays submitted per user
essay_count = essay_results_clean.groupby('user_id')['result_id'].count().reset_index(name='essay_count')

# Merge into perf_by_users
perf_by_users = perf_by_users.merge(avg_essay_score, on='user_id', how='left')
perf_by_users = perf_by_users.merge(essay_count, on='user_id', how='left')

perf_by_users.sample(10)

,user_id,average_point_quiz_absolute,average_point_quiz_relative,average_point_math_absolute,average_point_math_relative,average_essay_score,essay_count
2103,7104,0.259887,0.262857,NaN,NaN,NaN,NaN
1419,3849,NaN,NaN,NaN,NaN,3.631868,13.0
41,116,0.603922,0.608696,NaN,NaN,4.901786,8.0
963,1606,0.769231,0.769231,NaN,NaN,4.392857,2.0
1976,6896,0.830189,0.836538,NaN,NaN,NaN,NaN
452,956,0.555556,0.555556,NaN,NaN,4.571429,1.0
2035,7010,NaN,NaN,NaN,NaN,4.000000,1.0
2100,7100,0.288732,0.290780,NaN,NaN,NaN,NaN
1821,6540,0.635945,0.638889,NaN,NaN,4.085714,5.0
1554,6157,0.750000,0.750000,NaN,NaN,4.357143,3.0


Text comprehension scores

In [44]:
# Fix inconsistent rows
text_results.loc[text_results['points'] > text_results['max_points'], 'max_points'] = \
    text_results.loc[text_results['points'] > text_results['max_points'], 'points']

# Fix when max points = 0
text_valid = text_results[text_results['max_points'] > 0].copy()
text_valid['points_relative'] = text_valid['points'] / text_valid['max_points']

# Absolute and relative scores per user
avg_point_text_absolute = text_valid.groupby('user_id')['points'].mean().reset_index(name='average_point_text_absolute')
avg_point_text_relative = text_valid.groupby('user_id')['points_relative'].mean().reset_index(name='average_point_text_relative')

# Merge into perf_by_users
perf_by_users = perf_by_users.merge(avg_point_text_absolute, on='user_id', how='left')
perf_by_users = perf_by_users.merge(avg_point_text_relative, on='user_id', how='left')

perf_by_users.sample(10)

,user_id,average_point_quiz_absolute,average_point_quiz_relative,average_point_math_absolute,average_point_math_relative,average_essay_score,essay_count,average_point_text_absolute,average_point_text_relative
1003,1678,0.631922,0.636066,NaN,NaN,NaN,NaN,NaN,NaN
2038,7018,0.062500,0.062500,NaN,NaN,NaN,NaN,NaN,NaN
2103,7104,0.259887,0.262857,NaN,NaN,NaN,NaN,NaN,NaN
487,1007,0.657343,0.661972,NaN,NaN,NaN,NaN,NaN,NaN
422,917,0.500000,0.500000,0.285714,0.392857,4.657143,5.0,0.463918,0.137506
1068,1752,0.787879,0.838710,0.611111,0.444444,NaN,NaN,1.506494,0.628355
1063,1745,0.775362,0.778182,NaN,NaN,NaN,NaN,1.153846,0.441484
386,851,0.535714,0.535714,0.000000,0.000000,4.422078,11.0,NaN,NaN
1186,1889,0.632353,0.632353,NaN,NaN,4.634921,9.0,NaN,NaN
1801,6508,0.719149,0.719149,NaN,NaN,4.446429,4.0,NaN,NaN


A combined/overall performance metric per user

In [45]:
# Compute a combined overall performance metric per user
# Use relative scores only (comparable across different assessment types)
# Average across available relative scores (NaN ignored with mean)

perf_by_users['overall_performance'] = perf_by_users[[
    'average_point_quiz_relative',
    'average_point_text_relative',
    'average_essay_score',
    'average_point_math_relative'
]].mean(axis=1)

perf_by_users.sample(10)

,user_id,average_point_quiz_absolute,average_point_quiz_relative,average_point_math_absolute,average_point_math_relative,average_essay_score,essay_count,average_point_text_absolute,average_point_text_relative,overall_performance
509,1035,NaN,NaN,NaN,NaN,4.464286,2.0,NaN,NaN,4.464286
1399,3772,NaN,NaN,NaN,NaN,3.571429,1.0,NaN,NaN,3.571429
1581,6204,0.452381,0.452381,NaN,NaN,3.928571,2.0,NaN,NaN,2.190476
2073,7061,0.708333,0.708333,0.000000,0.000000,NaN,NaN,NaN,NaN,0.354167
2130,7143,0.740741,0.747664,NaN,NaN,4.047619,3.0,NaN,NaN,2.397641
109,290,NaN,NaN,NaN,NaN,4.792208,11.0,0.926829,0.431243,2.611725
1285,2022,0.539295,0.545205,NaN,NaN,4.125000,4.0,0.083333,0.035833,1.568680
1086,1773,0.664286,0.673913,0.111111,0.055556,3.488095,6.0,0.916667,0.359259,1.144206
689,1271,0.772512,0.776190,NaN,NaN,4.071429,1.0,NaN,NaN,2.423810
184,516,0.620690,0.620690,NaN,NaN,NaN,NaN,NaN,NaN,0.620690


---

## Start of Q1

In [54]:
# Q1: Chatbot Engagement Features
import ast

# 1. Features from gymitrainer.csv
gymitrainer = gymitrainer.rename(columns={'Unnamed: 0': 'thread_id'})

def parse_messages(content):
    try:
        if isinstance(content, str):
            messages = ast.literal_eval(content)
        else:
            messages = content
        student_msgs = [m for m in messages if not m['gymitrainer']]
        bot_msgs = [m for m in messages if m['gymitrainer']]
        return len(messages), len(student_msgs), len(bot_msgs)
    except:
        return 0, 0, 0

gymitrainer[['total_messages', 'student_messages', 'bot_messages']] = gymitrainer['content'].apply(
    lambda x: pd.Series(parse_messages(x))
)

# Aggregate per user
chatbot_features = gymitrainer.groupby('user_id').agg(
    chatbot_sessions=('thread_id', 'count'),
    total_messages=('total_messages', 'sum'),
    avg_messages_per_session=('total_messages', 'mean'),
    student_messages_total=('student_messages', 'sum'),
    bot_messages_total=('bot_messages', 'sum'),
).reset_index()

In [55]:
# 2. Features from gymitrainer_feedback.csv
feedback_features = gymitrainer_feedback.groupby('thread_id').agg(
    avg_feedback_score=('score', 'mean'),
    feedback_count=('score', 'count')
).reset_index()

# Link feedback to user via thread_id
gymitrainer_with_feedback = gymitrainer[['thread_id', 'user_id']].merge(
    feedback_features, on='thread_id', how='left'
)

user_feedback_features = gymitrainer_with_feedback.groupby('user_id').agg(
    avg_chatbot_feedback_score=('avg_feedback_score', 'mean'),
    total_feedback_given=('feedback_count', 'sum')
).reset_index()

In [56]:
# 3. Merge into perf_by_users
perf_by_users = perf_by_users.merge(chatbot_features, on='user_id', how='left')
perf_by_users = perf_by_users.merge(user_feedback_features, on='user_id', how='left')

perf_by_users.sample(10)

,user_id,average_point_quiz_absolute,average_point_quiz_relative,average_point_math_absolute,average_point_math_relative,average_essay_score,essay_count,average_point_text_absolute,average_point_text_relative,overall_performance,...,bot_messages_total_x,avg_chatbot_feedback_score_x,total_feedback_given_x,chatbot_sessions_y,total_messages_y,avg_messages_per_session_y,student_messages_total_y,bot_messages_total_y,avg_chatbot_feedback_score_y,total_feedback_given_y
1594,6247,0.291667,0.304348,NaN,NaN,4.928571,2.0,NaN,NaN,2.616460,...,2.0,NaN,0.0,1.0,4.0,4.000000,2.0,2.0,NaN,0.0
1981,6904,0.557692,0.572368,0.637931,0.465517,4.014286,5.0,NaN,NaN,1.684057,...,19.0,NaN,0.0,3.0,38.0,12.666667,19.0,19.0,NaN,0.0
1312,2235,0.538462,0.538462,NaN,NaN,NaN,NaN,NaN,NaN,0.538462,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1866,6617,0.857143,0.857143,NaN,NaN,4.738095,3.0,NaN,NaN,2.797619,...,5.0,NaN,0.0,1.0,12.0,12.000000,7.0,5.0,NaN,0.0
1972,6891,NaN,NaN,NaN,NaN,4.732143,8.0,NaN,NaN,4.732143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
530,1070,0.715736,0.719388,NaN,NaN,NaN,NaN,NaN,NaN,0.719388,...,209.0,NaN,0.0,16.0,419.0,26.187500,210.0,209.0,NaN,0.0
2117,7130,NaN,NaN,NaN,NaN,3.392857,2.0,NaN,NaN,3.392857,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
786,1389,0.561644,0.569444,NaN,NaN,NaN,NaN,NaN,NaN,0.569444,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692,6374,0.588235,0.588235,NaN,NaN,NaN,NaN,NaN,NaN,0.588235,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1607,6263,0.708171,0.708171,NaN,NaN,4.607143,2.0,NaN,NaN,2.657657,...,30.0,0.5,2.0,6.0,62.0,10.333333,32.0,30.0,0.5,2.0
